<a href="https://colab.research.google.com/github/Lutava2245/Allugator-Chatbot/blob/main/aligator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aligator, o chatbot de Allugator

Este é Aligator, um chatbot para responder perguntas relacionadas ao sistema de aluguéis de items Allugator.

### Configuração da API

In [17]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('FATEC_API_KEY'))

### Adicionando contexto ao Aligator

In [18]:
SYSTEM_INSTRUCTION = """
Você é Aligator, o assistente virtual oficial da Allugator, uma plataforma online de aluguel de itens entre usuários. Seu nome faz referência ao mascote da marca, um jacaré simpático.

Sua personalidade deve ser:
- amigável,
- clara,
- objetiva,
- prestativa,
- confiável,
- levemente descontraída, sem exageros.

### Sua missão

Responder apenas 3 perguntas do usuário sobre o funcionamento da plataforma Allugator, usando somente as informações deste contexto.
Se a pergunta estiver fora do escopo, informe educadamente que você só pode responder com base nas informações disponíveis neste contexto.

### Informações sobre a Allugator
- O usuário faz cadastro com nome, e-mail e senha.
- Após o cadastro, ele é redirecionado para a tela principal.
- A tela principal contém um dashboard de ganhos e gastos dentro do app.
- O usuário pode explorar um catálogo de itens disponíveis que foram postados por outros usuários.
- Se quiser alugar um item de outro usuário, ele:
1. escolhe um item no catálogo,
2. faz uma reserva,
3. pode se comunicar com o dono,
4. após a confirmação, adiciona um cartão de débito ou crédito,
5. e então recebe o item.
- Se quiser anunciar um item próprio para aluguel, ele cadastra:
1. nome do item,
2. descrição,
3. preço.

### Regras importantes
- Responda somente 3 perguntas válidas.
- Não invente funcionalidades que não estejam descritas.
- Não assuma políticas, taxas, avaliações, entrega detalhada ou recursos extras.
- Seja claro e breve.
"""

### Configurando o Aligator

In [19]:
from google.genai import types

conversation_history: list[types.Content] = []
question_count = 0
MAX_QUESTIONS = 3
MODEL = "gemini-2.5-flash"

def get_response(user_input: str) -> str:
    global question_count
    question_count += 1

    text = user_input
    if question_count >= MAX_QUESTIONS:
        text += (
            "\n\n[INSTRUÇÃO INTERNA — NÃO EXIBIR: "
            "Nesta 3ª pergunta, você deve"
            "responder a pergunta normalmente,"
            "apresentar um breve resumo do que foi explicado ao longo da conversa,"
            "informe que o limite de 3 perguntas foi atingido"
            "encerre a conversa de forma cordial."
        )

    conversation_history.append(
        types.Content(role="user", parts=[types.Part(text=text)])
    )

    response = client.models.generate_content(
        model=MODEL,
        config=types.GenerateContentConfig(
            system_instruction = SYSTEM_INSTRUCTION,
        ),
        contents = conversation_history,
    )

    assistant_text = response.text

    conversation_history.append(
        types.Content(
            role="model",
            parts=[types.Part(text=assistant_text)])
    )

    return assistant_text

### Implementando o chatbot

In [ ]:
conversation_history.clear()
question_count = 0

print(f"""
================================================================
🐊 Allugator | Aluguéis de itens
Oi! Sou o Aligator, assistente do Sistema de Aluguéis Allugator.
Respondo no máximo {MAX_QUESTIONS} perguntas nesta conversa.
================================================================
""")

while question_count < MAX_QUESTIONS:
    num = question_count + 1
    user_input = input(f"[{num}/{MAX_QUESTIONS}] Você: ").strip()
    if not user_input:
        continue
    print(f"\n🐊 Alugator:\n{get_response(user_input)}\n")

print("""
======================
Atendimento encerrado.
======================
""")